In [ ]:
#Bibioteki
import os
from PIL import Image
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt


In [ ]:
#Tworzenie wektorow z zdjec

#Scieżka do zdjec
dataset_path = '/content/drive/MyDrive/Final_images_dataset'

# Ładowanie CLIP
print("Ładowanie modelu CLIP...")
model = SentenceTransformer('clip-ViT-B-32')

# Wczytywanie obrazów
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
image_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(image_extensions)]
image_files.sort()

print(f"Znaleziono {len(image_files)} obrazów.")

# Generowanie wektorów
images_pil = []
valid_files = []

for file_name in image_files:
    try:
        img_path = os.path.join(dataset_path, file_name)
        img = Image.open(img_path).convert('RGB')
        images_pil.append(img)
        valid_files.append(file_name)
    except Exception as e:
        print(f"Nie udało się wczytać: {file_name} - {e}")

print("Przetwarzanie obrazów na wektory...")
embeddings = model.encode(images_pil)

print("Gotowe! Wektory zapisane")
print(f"Kształt macierzy wektorów: {embeddings.shape}")

Ładowanie modelu CLIP...
Znaleziono 104 obrazów.
Przetwarzanie obrazów na wektory...
Gotowe! Wektory zapisane
Kształt macierzy wektorów: (104, 512)


In [ ]:
#Szukanie oulayerów + znajdowanie ilosci kategori

#Przygotowanie danych
scaler = StandardScaler()
embeddings_normalized = scaler.fit_transform(embeddings)
#Zmniejsza wymiar macierzy wektorów
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(embeddings_normalized)

dataset_size = len(X_reduced)
# Dynamiczne min_samples (logarytm - 1)
calculated_min_samples = max(2, int(np.log(dataset_size)) - 1)

# Akceptowalny zasięg szumu by nie znalazło jednej kategori i 99 outlayerow
MIN_NOISE = 0.01
MAX_NOISE = 0.1

eps_range = np.arange(5.0, 50.0, 0.5)
best_eps = -1
best_score = -100


for eps_check in eps_range:
    # Szybki fit
    db = DBSCAN(eps=eps_check, min_samples=calculated_min_samples, metric='euclidean')
    labels = db.fit_predict(X_reduced)

    # Liczymy szum
    n_noise = list(labels).count(-1)
    noise_ratio = n_noise / dataset_size

   #sprawdzanie szumu
    if noise_ratio > MAX_NOISE or noise_ratio < MIN_NOISE:
        continue

    # czy wiecej niz jedna kategoria
    unique_labels = set(labels)
    if -1 in unique_labels: unique_labels.remove(-1)
    if len(unique_labels) < 2:
        continue

    # Liczenie jakosci podzialu
    mask = labels != -1
    if np.sum(mask) < 2: continue

    score = silhouette_score(X_reduced[mask], labels[mask])

    # Szukanie najlepszego wyniku
    if score > best_score:
        best_score = score
        best_eps = eps_check
#Jezeli nie ma dobrego wyniku bierze deafult
if best_eps == -1:
    print("Nie znaleziono idealnego kandydata w Safe Zone. Bierze domyślny fallback.")
    best_eps = 25.0

# 3. Uruchomienie z najlepszymi parametrami
dbscan_final = DBSCAN(eps=best_eps, min_samples=calculated_min_samples, metric='euclidean')
labels = dbscan_final.fit_predict(X_reduced)

n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print("-" * 40)
print(f"Liczba outlierów: {n_noise_}")
print("-" * 40)

if n_noise_ > 0:

    print("\nOUTLIERY:")
    outlier_indices = np.where(labels == -1)[0]
    for idx in outlier_indices:
        print(f" -> {valid_files[idx]}")

#Podzielenie na kategorie po usunieciu outlayerow

# Usuniecie outlayerow
clean_mask = labels != -1
X_clean = X_reduced[clean_mask]

# Konwersja listy plików na numpy array, żeby użyć maski
files_clean = np.array(valid_files)[clean_mask]

if len(X_clean) < 2:
    print("Za mało danych po usunięciu outlierów, by szukać kategorii.")
else:
    # Szukanie idealnej liczby kategori za pomoco kmeans
    max_k = min(10, len(X_clean))

    best_k = -1
    best_k_score = -1
    best_k_labels = []

    for k in range(2, max_k + 1):
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
        k_labels = kmeans.fit_predict(X_clean)
        score = silhouette_score(X_clean, k_labels)

        if score > best_k_score:
            best_k_score = score
            best_k = k
            best_k_labels = k_labels

    print(f"\nZnaleziony podział:")
    print(f"Liczba kategorii: {best_k}")

----------------------------------------
Liczba outlierów: 9
----------------------------------------

OUTLIERY:
 -> wno_03.jpg
 -> wno_04.jpg
 -> wno_06.jpg
 -> wno_08.jpg
 -> wno_15.jpeg
 -> wno_66.jpg
 -> wno_87.jpg
 -> wno_88.jpg
 -> wno_89.jpg

Znaleziony podział:
Liczba kategorii: 8
